# <font color="#418FDE" size="6.5" uppercase>**Images Custom Transformers**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Represent image data as arrays and extract patch-based features for modeling workflows. 
- Create graph-style image features using pixel neighborhoods and regular grid structures. 
- Implement custom transformers that follow Scikit-Learn fit and transform conventions. 


## **1. Image Patch Features**

### **1.1. Patch Feature Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_01_01.jpg?v=1783836691" width="250">



>* Images are numeric arrays of pixels.
>* Patches capture local patterns for modeling.

>* Patches become feature vectors for models.
>* Patch size and extraction settings shape context.

>* Patches turn images into model-ready tables.
>* They capture local details flexibly across tasks.



In [ ]:
#@title Python Code - Patch Feature Basics

# Images become arrays of numeric values.
# Small patches capture local visual patterns.
# Patch vectors support later modeling steps.

import numpy as np
import matplotlib.pyplot as plt

# Build a tiny grayscale image array.
image = np.array([
    [0, 0, 1, 1, 0, 0],
    [0, 1, 2, 2, 1, 0],
    [1, 2, 3, 3, 2, 1],

    [1, 2, 3, 3, 2, 1],
    [0, 1, 2, 2, 1, 0],
    [0, 0, 1, 1, 0, 0],
], dtype=float)

# Define a simple patch extractor.
def extract_patches(arr, patch_size=(2, 2), step=2):
    rows, cols = arr.shape
    ph, pw = patch_size

    # Validate patch settings safely.
    if ph > rows or pw > cols:
        raise ValueError("Patch size exceeds image size.")
    patches, positions = [], []

    # Slide a window across image.
    for r in range(0, rows - ph + 1, step):
        for c in range(0, cols - pw + 1, step):
            patch = arr[r:r + ph, c:c + pw]
            patches.append(patch.reshape(-1))

            positions.append((r, c))
    return np.array(patches), positions

# Extract non-overlapping two-by-two patches.
patch_vectors, patch_positions = extract_patches(
    image, patch_size=(2, 2), step=2
)

# Show key shape information briefly.
print("Image shape:", image.shape)
print("Patch matrix shape:", patch_vectors.shape)
print("First patch position:", patch_positions[0])
print("First patch vector:", patch_vectors[0].tolist())

# Plot image and patch boundaries.
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(image, cmap="gray", vmin=0, vmax=3)

# Draw patch grid lines clearly.
for edge in [1.5, 3.5]:
    ax.axhline(edge, color="red", linewidth=1)
    ax.axvline(edge, color="red", linewidth=1)

# Add a short explanatory title.
ax.set_title("6x6 image split into 2x2 patches")
ax.set_xticks(range(6)); ax.set_yticks(range(6))
plt.tight_layout(); plt.show()



### **1.2. Patch Feature Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_01_02.jpg?v=1783836683" width="250">



>* Images are numeric arrays with spatial structure.
>* Patches capture local patterns for modeling.

>* Sliding windows create local patch features.
>* Patches preserve context for reliable prediction.

>* Patch size, overlap, and normalization shape features.
>* Patches preserve local patterns for modeling.



In [ ]:
#@title Python Code - Patch Feature Basics

# Images can become structured numeric arrays.
# Small patches capture local visual patterns.
# This example extracts simple patch features.

import numpy as np
import matplotlib.pyplot as plt

# Build a tiny grayscale image.
image = np.arange(36, dtype=float).reshape(6, 6)
# Add a brighter square region.
image[2:4, 2:5] += 20

# Set patch size and stride.
patch_h, patch_w = 2, 2
stride = 2

# Collect flattened patch vectors.
patch_vectors = []
patch_means = []
positions = []

# Slide a window across image.
for row in range(0, image.shape[0] - patch_h + 1, stride):
    for col in range(0, image.shape[1] - patch_w + 1, stride):
        patch = image[row:row + patch_h, col:col + patch_w]
        patch_vectors.append(patch.ravel())
        patch_means.append(patch.mean())
        positions.append((row, col))

# Convert lists into arrays.
patch_vectors = np.array(patch_vectors)
patch_means = np.array(patch_means)
positions = np.array(positions)

# Check extracted feature shapes.
assert patch_vectors.shape[1] == patch_h * patch_w
assert len(patch_means) == len(positions)

# Print a compact feature summary.
print("Image shape:", image.shape)
print("Patch vector shape:", patch_vectors.shape)
print("First patch position:", tuple(positions[0]))
print("First patch vector:", patch_vectors[0].astype(int).tolist())
print("Patch mean features:", np.round(patch_means, 1).tolist())

# Show image and patch grid.
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(image, cmap="gray", interpolation="nearest")

# Draw patch boundaries clearly.
for row, col in positions:
    rect = plt.Rectangle(
        (col - 0.5, row - 0.5),
        patch_w,
        patch_h,
        fill=False,
        edgecolor="red",
        linewidth=2,
    )
    ax.add_patch(rect)

# Add a short plot title.
ax.set_title("2x2 image patches")
ax.set_xticks(range(image.shape[1]))
ax.set_yticks(range(image.shape[0]))
plt.tight_layout()
plt.show()



### **1.3. Patch Feature Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_01_03.jpg?v=1783836695" width="250">



>* Images are numeric arrays of pixels.
>* Patches capture local patterns as features.

>* Patches reveal local patterns beyond single pixels.
>* Patch features preserve detail for tabular models.

>* Patch size and overlap shape captured detail.
>* Patches become consistent features for pipelines.



In [ ]:
#@title Python Code - Patch Feature Basics

# Images can become useful numeric feature tables.
# Small patches capture local visual structure.
# This example shows basic patch extraction.

import numpy as np
import matplotlib.pyplot as plt

# Build a tiny grayscale image array.
image = np.array([
    [0, 0, 1, 1, 0, 0],
    [0, 1, 2, 2, 1, 0],
    [1, 2, 3, 3, 2, 1],
    [1, 2, 3, 3, 2, 1],
    [0, 1, 2, 2, 1, 0],
    [0, 0, 1, 1, 0, 0],
], dtype=float)

# Check the image shape safely.
if image.shape != (6, 6):
    raise ValueError("Expected a 6 by 6 image array.")

# Extract overlapping square patches.
patch_size = 2
patches = []

# Slide across rows and columns.
for row in range(image.shape[0] - patch_size + 1):
    for col in range(image.shape[1] - patch_size + 1):
        patch = image[row:row + patch_size, col:col + patch_size]
        patches.append(patch)

# Convert patches into one array.
patches = np.array(patches)
flat_patches = patches.reshape(patches.shape[0], -1)

# Create simple patch summary features.
patch_means = flat_patches.mean(axis=1)
patch_ranges = flat_patches.max(axis=1) - flat_patches.min(axis=1)
feature_table = np.column_stack((patch_means, patch_ranges))

# Print a few compact results.
print("Image shape:", image.shape)
print("Patch array shape:", patches.shape)
print("Flattened patch shape:", flat_patches.shape)
print("Feature table shape:", feature_table.shape)
print("First patch values:", flat_patches[0].tolist())
print("First patch features:", feature_table[0].round(2).tolist())

# Plot the image and one patch.
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(image, cmap="gray", vmin=0, vmax=3)
axes[0].set_title("Full image")
axes[0].axis("off")

# Show the first extracted patch.
axes[1].imshow(patches[0], cmap="gray", vmin=0, vmax=3)
axes[1].set_title("First 2x2 patch")
axes[1].axis("off")

# Finish the single teaching plot.
plt.tight_layout()
plt.show()



## **2. Image Graph Features**

### **2.1. Pixel Neighborhood Graphs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_02_01.jpg?v=1783836689" width="250">



>* Pixels become nodes linked to neighbors.
>* Graphs preserve local structure for image features.

>* Visual patterns come from neighboring pixel relationships.
>* Graphs reveal similarity, boundaries, and connectivity.

>* Neighborhood size changes detail versus context.
>* Weighted connections capture similarity for structured features.



In [ ]:
#@title Python Code - Pixel Neighborhood Graphs

# Pixels can become graph nodes.
# Neighbor links preserve local structure.
# This example builds image graph features.

import numpy as np
import matplotlib.pyplot as plt

# Create a tiny grayscale image.
image = np.array([
    [0.10, 0.10, 0.15, 0.80],
    [0.10, 0.20, 0.25, 0.85],

    [0.12, 0.22, 0.70, 0.90],
    [0.15, 0.25, 0.75, 0.95],
], dtype=float)

# Check the image shape.
height, width = image.shape
assert height > 1 and width > 1

# Convert row and column into node ids.
def node_id(row, col):
    return row * width + col

# Build four-neighbor graph edges.
edges = []
for row in range(height):
    for col in range(width):
        current = node_id(row, col)
        if col + 1 < width:

            right = node_id(row, col + 1)
            weight = abs(image[row, col] - image[row, col + 1])
            edges.append((current, right, weight))
        if row + 1 < height:

            down = node_id(row + 1, col)
            weight = abs(image[row, col] - image[row + 1, col])
            edges.append((current, down, weight))

# Summarize graph style features.
weights = np.array([edge[2] for edge in edges])
node_count = height * width
edge_count = len(edges)

# Count strong local changes.
strong_edges = int(np.sum(weights > 0.30))
average_change = float(np.mean(weights))
max_change = float(np.max(weights))

# Print a compact explanation.
print(f"Image shape: {height} x {width}")
print(f"Nodes: {node_count}")
print(f"Edges: {edge_count}")
print(f"Average neighbor change: {average_change:.2f}")

print(f"Largest neighbor change: {max_change:.2f}")
print(f"Strong edges over 0.30: {strong_edges}")
print("Bright boundary creates larger edge weights.")

# Plot the image values.
plt.figure(figsize=(4, 4))
plt.imshow(image, cmap="gray", vmin=0, vmax=1)
plt.title("Tiny image for neighborhood graph")

# Label each pixel value.
for row in range(height):
    for col in range(width):
        plt.text(
            col, row, f"{image[row, col]:.2f}",
            ha="center", va="center", color="red"
        )

# Show the single allowed plot.
plt.xticks(range(width))
plt.yticks(range(height))
plt.grid(color="white", linewidth=1)
plt.show()



### **2.2. Pixel Neighborhood Graphs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_02_02.jpg?v=1783836701" width="250">



>* Pixels become nodes linked to neighbors.
>* Graphs reveal edges, boundaries, and textures.

>* Neighborhood choice shapes captured image structure.
>* Weights reveal smooth areas and boundaries.

>* Local similarity reveals surfaces, edges, and textures.
>* Graph features aid segmentation and defect detection.



In [ ]:
#@title Python Code - Pixel Neighborhood Graphs

# Pixel graphs capture local image structure.
# This example builds neighborhood connections.
# We summarize edges with simple features.
# pip install matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt

# Create a tiny grayscale image.
image = np.array([
    [0.10, 0.10, 0.10, 0.10, 0.10],
    [0.10, 0.80, 0.80, 0.10, 0.10],
    [0.10, 0.80, 0.20, 0.10, 0.10],
    [0.10, 0.10, 0.10, 0.10, 0.10],
    [0.10, 0.10, 0.10, 0.90, 0.90],
], dtype=float)

# Check the image shape safely.
assert image.ndim == 2 and image.size > 0

# Build four-neighbor graph features.
def pixel_graph_features(arr):

    # Validate a two-dimensional input array.
    assert arr.ndim == 2 and arr.shape[0] > 1

    rows, cols = arr.shape
    edge_count = 0
    total_difference = 0.0

    # Store local contrast per pixel.
    local_contrast = np.zeros_like(arr)

    # Visit each pixel once.
    for r in range(rows):
        for c in range(cols):

            # Compare right and lower neighbors.
            for dr, dc in ((0, 1), (1, 0)):
                nr, nc = r + dr, c + dc

                # Skip neighbors outside bounds.
                if nr >= rows or nc >= cols:
                    continue

                diff = abs(arr[r, c] - arr[nr, nc])
                edge_count += 1
                total_difference += diff

                # Add contrast to both nodes.
                local_contrast[r, c] += diff
                local_contrast[nr, nc] += diff

    # Compute compact graph summaries.
    mean_edge_difference = total_difference / edge_count
    strongest_pixel = np.unravel_index(local_contrast.argmax(), arr.shape)

    return local_contrast, edge_count, mean_edge_difference, strongest_pixel

# Extract graph-style image features.
contrast_map, edges, mean_diff, strongest = pixel_graph_features(image)

# Print a short explanation.
print(f"Image shape: {image.shape}")
print(f"Graph edges: {edges}")
print(f"Mean neighbor difference: {mean_diff:.3f}")
print(f"Strongest local contrast pixel: {strongest}")

# Plot image and graph feature map.
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

# Show the original pixel grid.
axes[0].imshow(image, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Original image")
axes[0].set_xticks(range(image.shape[1]))
axes[0].set_yticks(range(image.shape[0]))

# Show neighborhood contrast feature.
axes[1].imshow(contrast_map, cmap="magma")
axes[1].set_title("Pixel graph feature")
axes[1].set_xticks(range(image.shape[1]))
axes[1].set_yticks(range(image.shape[0]))

plt.tight_layout()
plt.show()



### **2.3. Grid Connectivity**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_02_03.jpg?v=1783836703" width="250">



>* Pixels form graphs through local grid neighbors.
>* This preserves spatial patterns for image tasks.

>* Connectivity choice shapes captured image structures.
>* Weighted neighbors reveal regions, boundaries, textures.

>* Grid graphs enable structural image features.
>* Useful for defects, regions, and objects.



In [ ]:
#@title Python Code - Grid Connectivity

# Grid graphs preserve local image structure.
# This example compares two neighborhood rules.
# A tiny image keeps everything readable.

import numpy as np
import matplotlib.pyplot as plt

# Build a small binary image.
image = np.array([
    [0, 1, 1, 0],
    [0, 1, 0, 0],
    [1, 1, 0, 1],
    [0, 0, 1, 1],
], dtype=int)

# Map each pixel to one node.
rows, cols = image.shape
node_ids = np.arange(rows * cols).reshape(rows, cols)

# Define neighbor offsets clearly.
four_offsets = [(-1, 0), (1, 0), (0, -1), (0, 1)]
eight_offsets = four_offsets + [(-1, -1), (-1, 1), (1, -1), (1, 1)]

# Collect undirected graph edges safely.
def build_edges(offsets):

    edges = set()
    for r in range(rows):
        for c in range(cols):
            for dr, dc in offsets:
                nr, nc = r + dr, c + dc

                inside = 0 <= nr < rows and 0 <= nc < cols
                if inside:
                    a = int(node_ids[r, c])
                    b = int(node_ids[nr, nc])
                    edges.add(tuple(sorted((a, b))))

    return sorted(edges)

# Build both connectivity patterns.
edges4 = build_edges(four_offsets)
edges8 = build_edges(eight_offsets)

# Count bright neighboring pairs.
def bright_edge_count(edges):

    total = 0
    for a, b in edges:
        ra, ca = divmod(a, cols)
        rb, cb = divmod(b, cols)

        if image[ra, ca] == 1 and image[rb, cb] == 1:
            total += 1

    return total

# Summarize graph style features.
bright4 = bright_edge_count(edges4)
bright8 = bright_edge_count(edges8)
node_count = rows * cols

# Print a short interpretation.
print(f"Image shape: {rows} x {cols}")
print(f"Nodes in graph: {node_count}")
print(f"Four-neighbor edges: {len(edges4)}")
print(f"Eight-neighbor edges: {len(edges8)}")
print(f"Bright pairs with four-neighbors: {bright4}")
print(f"Bright pairs with eight-neighbors: {bright8}")

# Plot the image with node labels.
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(image, cmap="gray_r", vmin=0, vmax=1)

# Add node numbers onto pixels.
for r in range(rows):
    for c in range(cols):
        ax.text(c, r, str(node_ids[r, c]),
                ha="center", va="center", color="red")

# Finish the single teaching plot.
ax.set_title("Pixel nodes on a regular grid")
ax.set_xticks(range(cols))
ax.set_yticks(range(rows))
plt.tight_layout()
plt.show()



## **3. Building Custom Transformers**

### **3.1. Transformer Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_03_01.jpg?v=1783836705" width="250">



>* Reusable transformers standardize image preprocessing outputs.
>* They ensure consistent pipelines across workflow stages.

>* Transformers need clear, consistent input and output.
>* Simple interfaces improve reuse, testing, and teamwork.

>* Some transformers learn; others just apply rules.
>* Standards enable reliable, reusable production workflows.



In [ ]:
#@title Python Code - Transformer Basics

# Learn simple custom transformer structure here.
# Use tiny image arrays for clarity.
# Follow fit and transform conventions carefully.

import numpy as np

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create tiny grayscale image samples.
images = rng.integers(0, 256, size=(3, 4, 4))

# Define a beginner friendly transformer.
class PatchMeanTransformer:

    # Store patch size during setup.
    def __init__(self, patch_size=(2, 2)):
        self.patch_size = patch_size
        self.is_fitted_ = False

    # Learn expected image shape.
    def fit(self, X, y=None):
        X = np.asarray(X)
        if X.ndim != 3:
            raise ValueError("Expected shape: samples, height, width")

        # Save shape information safely.
        self.image_shape_ = X.shape[1:]
        self.is_fitted_ = True
        return self

    # Convert images into patch features.
    def transform(self, X):
        X = np.asarray(X)
        if not self.is_fitted_:
            raise ValueError("Call fit before transform")

        # Check incoming image shape.
        if X.shape[1:] != self.image_shape_:
            raise ValueError("Image shape differs from fitted data")
        ph, pw = self.patch_size

        # Validate patch compatibility.
        h, w = self.image_shape_
        if h % ph or w % pw:
            raise ValueError("Patch size must divide image dimensions")

        # Build one feature row per image.
        features = []
        for image in X:
            row = []
            for i in range(0, h, ph):
                for j in range(0, w, pw):
                    patch = image[i:i + ph, j:j + pw]
                    row.append(patch.mean())
            features.append(row)

        # Return a numeric feature matrix.
        return np.array(features, dtype=float)

# Create and use the transformer.
transformer = PatchMeanTransformer(patch_size=(2, 2))
transformer.fit(images)

# Transform images into features.
patch_features = transformer.transform(images)

# Show compact learning focused results.
print("Input image shape:", images.shape)
print("Learned image shape:", transformer.image_shape_)
print("Output feature shape:", patch_features.shape)
print("First image patch means:", np.round(patch_features[0], 1))



### **3.2. Fit Transform Methods**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_03_02.jpg?v=1783836711" width="250">



>* Fit learns settings from training data.
>* Transform applies them consistently to new images.

>* fit_transform combines learning and conversion efficiently
>* It must use only fitting-stage information

>* Fit once to avoid data leakage.
>* Transform consistently for reliable, reproducible features.



In [ ]:
#@title Python Code - Fit Transform Methods

# Custom transformers learn state during fitting.
# Transform reuses learned values consistently later.
# This example uses tiny image arrays.

import numpy as np

# Build a tiny image dataset.
train_images = np.array([
    [[1, 2], [3, 4]],
    [[2, 2], [2, 6]],
])

# Create new images for later transformation.
test_images = np.array([
    [[1, 1], [1, 5]],
    [[3, 3], [3, 3]],
])

# Define a simple custom transformer.
class MeanCenterPatchTransformer:
    def __init__(self):
        self.mean_ = None
        self.is_fitted_ = False

    # Learn one value from training images.
    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.mean_ = X.mean()
        self.is_fitted_ = True
        return self

    # Apply learned value to new images.
    def transform(self, X):
        if not self.is_fitted_:
            raise ValueError("Call fit before transform.")
        X = np.asarray(X, dtype=float)
        centered = X - self.mean_
        return centered.reshape(len(X), -1)

    # Combine fitting and transforming conveniently.
    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X)

# Fit and transform training images.
transformer = MeanCenterPatchTransformer()
train_features = transformer.fit_transform(train_images)

# Transform test images without relearning.
test_features = transformer.transform(test_images)

# Show the learned state and shapes.
print("Learned mean:", round(transformer.mean_, 2))
print("Train feature shape:", train_features.shape)
print("Test feature shape:", test_features.shape)
print("First train feature row:", np.round(train_features[0], 2))
print("First test feature row:", np.round(test_features[0], 2))



### **3.3. Transformer Fit Methods**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_03_03.jpg?v=1783836713" width="250">



>* Fit learns and stores training-based settings.
>* It may only validate and return itself.

>* Fit learns only from training data.
>* Fit validates inputs and prevents leakage.

>* Fit stores learned state, not outputs.
>* This separation supports reusable pipeline transformers.



In [ ]:
#@title Python Code - Transformer Fit Methods

# Custom transformers can learn image statistics.
# Fit stores values for later transforms.
# This example uses tiny image patches.

import numpy as np

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create tiny grayscale training images.
train_images = rng.integers(0, 256, size=(4, 4, 4))

# Define a simple transformer class.
class PatchMeanCenterer:

    # Store patch size during setup.
    def __init__(self, patch_size=(2, 2)):
        self.patch_size = patch_size
        self.patch_means_ = None

    # Validate image array carefully.
    def _check_images(self, X):
        X = np.asarray(X, dtype=float)
        if X.ndim != 3:
            raise ValueError("Expected shape: samples, height, width")

        # Check patch compatibility safely.
        h, w = X.shape[1], X.shape[2]
        ph, pw = self.patch_size
        if h % ph or w % pw:
            raise ValueError("Image size must match patch grid")
        return X

    # Learn patch means from training data.
    def fit(self, X, y=None):
        X = self._check_images(X)
        ph, pw = self.patch_size
        means = []

        # Average each patch position.
        for row in range(0, X.shape[1], ph):
            for col in range(0, X.shape[2], pw):
                patch = X[:, row:row + ph, col:col + pw]
                means.append(patch.mean())

        # Save learned state and shape.
        self.patch_means_ = np.array(means)
        self.image_shape_ = X.shape[1:]
        return self

    # Apply learned means to new images.
    def transform(self, X):
        X = self._check_images(X)
        if self.patch_means_ is None:
            raise ValueError("Call fit before transform")

        # Confirm incoming image shape.
        if tuple(X.shape[1:]) != tuple(self.image_shape_):
            raise ValueError("New images must match fitted shape")
        ph, pw = self.patch_size
        features = []

        # Build centered patch features.
        for image in X:
            image_features = []
            index = 0
            for row in range(0, image.shape[0], ph):
                for col in range(0, image.shape[1], pw):
                    patch = image[row:row + ph, col:col + pw]
                    centered = patch.mean() - self.patch_means_[index]
                    image_features.append(centered)
                    index += 1

            # Save one feature vector.
            features.append(image_features)
        return np.array(features)

# Create one new test image.
test_images = rng.integers(0, 256, size=(1, 4, 4))

# Fit on training images only.
transformer = PatchMeanCenterer(patch_size=(2, 2)).fit(train_images)

# Transform the new image.
patch_features = transformer.transform(test_images)

# Show what fit learned.
print("Training image shape:", transformer.image_shape_)
print("Learned patch means:", np.round(transformer.patch_means_, 1))
print("Output feature shape:", patch_features.shape)
print("Centered patch features:", np.round(patch_features[0], 1))



# <font color="#418FDE" size="6.5" uppercase>**Images Custom Transformers**</font>


In this lecture, you learned to:
- Represent image data as arrays and extract patch-based features for modeling workflows. 
- Create graph-style image features using pixel neighborhoods and regular grid structures. 
- Implement custom transformers that follow Scikit-Learn fit and transform conventions. 

In the next Module (Module 8), we will go over 'None'